# FenriX — Identification Benchmark (collect & grade)

Runs each local model over all 8 anonymized 10-Ks and records its **company guess + reasoning + tokens + latency**. No auto-scoring — you grade the final table by eye against your own answer key.

**How it works:** for each `(model, thinking-mode)` it launches `llama-server` (native GPU) via `../models/scripts/serve.sh`, sends each report through the identification prompt, and saves raw answers to `../models/results/identifications/<model>__<mode>.json`. A final cell consolidates everything into `_grading.csv`.

**Prereqs:** models downloaded (`../models/scripts/download.sh`) and native llama.cpp built (`~/.local/bin/llama-server`).


In [ ]:
# Ensure deps (safe to re-run)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyyaml", "tqdm"])
print("deps ok")


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os, json, time, subprocess, urllib.request
from pathlib import Path
import yaml

NB_DIR     = Path.cwd()                                  # notebooks/
MODELS_DIR = (NB_DIR / ".." / "models").resolve()
DATA_DIR   = (NB_DIR / ".." / "data" / "FenriX_Synthetic_challenge").resolve()
OUT_DIR    = MODELS_DIR / "results" / "identifications"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PORT = 8080
CTX  = 16384          # reports are ~8-10k tokens; the registry's 8192 is too small

# Models to run (registry ids). llama33-70b is offload-slow (~3 tok/s) -> skipped by default.
REG        = yaml.safe_load(open(MODELS_DIR / "registry.yaml"))
ALL_MODELS = [m["id"] for m in REG["models"]]
SKIP       = {"llama33-70b"}                             # remove to include the slow 70B
MODELS     = [m for m in ALL_MODELS if m not in SKIP]

# Thinking dimension. "nothink" = --reasoning-budget 0 (crisp). "think" = default (reasons first).
THINKING_MODES = ["nothink"]                             # add "think" to test if reasoning helps ID
MAX_TOKENS     = {"nothink": 700, "think": 4000}

COMPANIES = sorted(p.name for p in DATA_DIR.iterdir()
                   if (p / "unstructured" / "ANNUAL_REPORT.txt").exists())

PROMPT_TMPL = (
    "You are given an anonymized annual report (Form 10-K). All company, product, "
    "person, and place names have been replaced with fictional ones, and dollar "
    "figures are illustrative. Based on the business description, sector, product/"
    "segment lineup, competitive positioning, and financial profile, identify the "
    "REAL publicly-traded company this report is most likely modeled on.\n\n"
    "Answer with:\n1) Your single best guess (one real company name).\n"
    "2) 3-5 specific clues from the text that led you there.\n\n"
    "=== REPORT ===\n{report}"
)

print("models   :", MODELS)
print("modes    :", THINKING_MODES)
print("companies:", len(COMPANIES), COMPANIES)


In [ ]:
# ── Helpers: server lifecycle + one identification call ──────────────────────
LLAMA_ENV = {**os.environ, "PATH": f"{Path.home()}/.local/bin:" + os.environ.get("PATH", "")}

def start_server(model_id, mode):
    """serve.sh exec's llama-server (native GPU). --reasoning-budget 0 disables thinking."""
    args = ["bash", str(MODELS_DIR / "scripts" / "serve.sh"), model_id, "--ctx-size", str(CTX)]
    if mode == "nothink":
        args += ["--reasoning-budget", "0"]
    log = open(OUT_DIR / f"_server_{model_id}__{mode}.log", "w")
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
                            env=LLAMA_ENV, cwd=str(MODELS_DIR))
    return proc, log

def wait_ready(port, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

def stop_server(proc, log):
    proc.terminate()
    try:    proc.wait(timeout=20)
    except Exception: proc.kill()
    log.close()

def identify(port, report_path, mode):
    report = Path(report_path).read_text(encoding="utf-8", errors="replace")
    body = json.dumps({
        "messages": [{"role": "user", "content": PROMPT_TMPL.format(report=report)}],
        "temperature": 0.3, "max_tokens": MAX_TOKENS[mode],
    }).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 data=body, headers={"Content-Type": "application/json"})
    t0 = time.time()
    resp = json.load(urllib.request.urlopen(req, timeout=900))
    dt = time.time() - t0
    ch = resp["choices"][0]; m = ch.get("message", {}); u = resp.get("usage", {})
    return {
        "guess":             (m.get("content") or "").strip(),
        "reasoning":         (m.get("reasoning_content") or "").strip(),
        "finish_reason":     ch.get("finish_reason"),
        "prompt_tokens":     u.get("prompt_tokens"),
        "completion_tokens": u.get("completion_tokens"),
        "latency_s":         round(dt, 1),
    }

print("helpers ready")


In [ ]:
# ── Run: for each (mode, model) -> serve -> all companies -> save json ───────
from tqdm.notebook import tqdm

def run_model(model_id, mode):
    out_path = OUT_DIR / f"{model_id}__{mode}.json"
    proc, log = start_server(model_id, mode)
    try:
        if not wait_ready(PORT):
            print(f"  server not ready for {model_id}/{mode} - skipping (see {log.name})")
            return None
        rows = []
        for comp in tqdm(COMPANIES, desc=f"{model_id}/{mode}", leave=False):
            rp = DATA_DIR / comp / "unstructured" / "ANNUAL_REPORT.txt"
            try:
                r = identify(PORT, rp, mode)
            except Exception as e:
                r = {"guess": "", "reasoning": "", "error": str(e)}
            r.update({"model": model_id, "mode": mode, "company": comp})
            rows.append(r)
        json.dump(rows, open(out_path, "w"), indent=2)
        return rows
    finally:
        stop_server(proc, log)

for mode in THINKING_MODES:
    for model_id in MODELS:
        print(f"> {model_id} / {mode}")
        rows = run_model(model_id, mode)
        if rows is not None:
            ok = sum(1 for r in rows if r.get("guess"))
            print(f"  {ok}/{len(rows)} answered -> identifications/{model_id}__{mode}.json")
print("done.")


In [ ]:
# ── Consolidate every identification into one grading table ──────────────────
import pandas as pd, glob

recs = []
for f in sorted(glob.glob(str(OUT_DIR / "*__*.json"))):
    recs.extend(json.load(open(f)))

df = pd.DataFrame(recs)
if df.empty:
    print("no identifications yet - run the cell above first")
else:
    cols = ["model", "mode", "company", "guess", "reasoning",
            "completion_tokens", "latency_s", "finish_reason"]
    df = df[[c for c in cols if c in df.columns]]
    grade = df.copy()
    grade["correct"] = ""   # you fill: 1 / 0
    grade["notes"]   = ""
    grade.to_csv(OUT_DIR / "_grading.csv", index=False)
    print(f"wrote {OUT_DIR/'_grading.csv'}  ({len(grade)} rows)")

    def short(s, n=90):
        s = (s or "").replace("\n", " ")
        return s[:n] + ("..." if len(s) > n else "")
    view = df.copy(); view["guess"] = view["guess"].map(short)
    display(view[["model", "mode", "company", "guess", "completion_tokens", "latency_s"]])


## Grading

Open `../models/results/identifications/_grading.csv`, fill the `correct` column (1/0) against your answer key, and re-load to compute accuracy. Each `<model>__<mode>.json` holds the full guess + reasoning.

**Dimensions to compare once graded:**
- **provider** — Gemma 4 vs Qwen3.5 vs Llama
- **size tier** — big-MoE vs mid-dense vs small-fast
- **thinking** — add `"think"` to `THINKING_MODES` and re-run to test whether reasoning improves identification (costs many more tokens)

Cross-reference speed from `../models/results/_summary.txt`.
